In [3]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
    pool_recycle=300
)

df = pd.read_sql(
    "SELECT * FROM gold.trip_analytics",
    engine
)

print(df.shape)

(174724, 37)


In [4]:
route_data = df.dropna(
    subset=["start_station_name", "end_station_name"]
).copy()

print(route_data.shape)

(174724, 37)


In [5]:
route_counts = (
    route_data
    .groupby(
        ["start_station_name", "end_station_name"]
    )
    .size()
    .reset_index(name="trip_count")
)

route_counts.head()

,start_station_name,end_station_name,trip_count
0,10th Ave at E 15th St,10th Ave at E 15th St,2
1,10th Ave at E 15th St,10th St at Fallon St,4
2,10th Ave at E 15th St,13th St at Franklin St,1
3,10th Ave at E 15th St,21st Ave at International Blvd,1
4,10th Ave at E 15th St,23rd Ave at Foothill Blvd,1


In [6]:
top_destinations = (
    route_counts
    .sort_values(
        ["start_station_name", "trip_count"],
        ascending=[True, False]
    )
    .drop_duplicates("start_station_name")
    .reset_index(drop=True)
)

top_destinations = top_destinations[
    ["start_station_name", "end_station_name", "trip_count"]
]

top_destinations.head(10)

,start_station_name,end_station_name,trip_count
0,10th Ave at E 15th St,Lake Merritt BART Station,9
1,10th St at Fallon St,2nd Ave at E 18th St,110
2,10th St at University Ave,North Berkeley BART Station,25
3,11th St at Bryant St,San Francisco Caltrain Station 2 (Townsend St...,76
4,11th St at Natoma St,San Francisco Caltrain Station 2 (Townsend St...,100
5,13th St at Franklin St,Lakeside Dr at 14th St,22
6,14th St at Filbert St,West Oakland BART Station,58
7,14th St at Mandela Pkwy,West Oakland BART Station,154
8,14th St at Mission St,San Francisco Caltrain Station 2 (Townsend St...,65
9,15th St at Potrero Ave,16th St Mission BART Station 2,55


In [7]:
top_dest_map = dict(
    zip(
        top_destinations["start_station_name"],
        top_destinations["end_station_name"]
    )
)

list(top_dest_map.items())[:10]

[('10th Ave at E 15th St', 'Lake Merritt BART Station'),
 ('10th St at Fallon St', '2nd Ave at E 18th St'),
 ('10th St at University Ave', 'North Berkeley BART Station'),
 ('11th St at Bryant St',
  'San Francisco Caltrain Station 2  (Townsend St at 4th St)'),
 ('11th St at Natoma St',
  'San Francisco Caltrain Station 2  (Townsend St at 4th St)'),
 ('13th St at Franklin St', 'Lakeside Dr at 14th St'),
 ('14th St at Filbert St', 'West Oakland BART Station'),
 ('14th St at Mandela Pkwy', 'West Oakland BART Station'),
 ('14th St at Mission St',
  'San Francisco Caltrain Station 2  (Townsend St at 4th St)'),
 ('15th St at Potrero Ave', '16th St Mission BART Station 2')]

In [8]:
from pathlib import Path

output_path = Path("Top_Destination.csv")

top_destinations.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path.resolve()}")

Saved to: C:\DiskD\AI\Depi-Projects-AI-upload\Data-Analysis\ford_gobike_analysis\Gold_DF\top_destination\Top_Destination.csv
